In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, current_timestamp, from_json
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text(
    "target_table", "raw_nasdaq_ticks", "4. Target Delta Table"
)
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "5. Secret Scope")
dbutils.widgets.text(
    "secret_key_eventhub", "valerii-eventhub-cs", "6. Event Hub Secret Key"
)
dbutils.widgets.text(
    "eventhub_name", "valeriimatviiv_evh", "7. Event Hub Entity Name"
)

# Retrieve parameter values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
target_table_name = dbutils.widgets.get("target_table")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key_eventhub = dbutils.widgets.get("secret_key_eventhub")
event_hub_name = dbutils.widgets.get("eventhub_name")

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
checkpoint_path = f"{base_path}/_state/checkpoints/nasdaq_ticks"
full_table_path = f"{catalog}.{schema}.{target_table_name}"

# Retrieve Event Hub connection string
connection_string = dbutils.secrets.get(
    scope=secret_scope, key=secret_key_eventhub
).strip()

# Build Kafka bootstrap server and SASL configuration
host_domain = connection_string.split("sb://")[1].split("/")[0]
BOOTSTRAP_SERVERS = f"{host_domain}:9093"

eh_sasl = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
    f'required username="$ConnectionString" password="{connection_string}";'
)

# 1. Read Stream from Event Hubs (using Kafka endpoint)
df_raw_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", event_hub_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("startingOffsets", "earliest")  # Reads existing events in queue
    .load()
)

# Schema for parsing price tick payload
tick_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("high_price", DoubleType(), True),
    StructField("low_price", DoubleType(), True),
    StructField("open_price", DoubleType(), True),
    StructField("previous_close", DoubleType(), True),
    StructField("event_timestamp", StringType(), True),
])

# 2. Parse Payload and Add Audit Metadata
df_parsed_stream = df_raw_stream.select(
    from_json(col("value").cast("string"), tick_schema).alias("data"),
    col("timestamp").alias("event_hub_enqueued_at"),
).select(
    col("data.symbol"),
    col("data.current_price"),
    col("data.high_price"),
    col("data.low_price"),
    col("data.open_price"),
    col("data.previous_close"),
    col("data.event_timestamp").cast("timestamp").alias("event_timestamp"),
    col("event_hub_enqueued_at"),
    current_timestamp().alias("_ingested_at"),
)

# 3. Write Stream to Delta Bronze Table
query = (
    df_parsed_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(full_table_path)
)

query.awaitTermination()

print(f"Streaming execution completed for Delta table: {full_table_path}")

In [0]:
# Databricks notebook source
display(spark.sql(f"SELECT * FROM {full_table_path} ORDER BY _ingested_at DESC"))
display(spark.sql(f"SELECT COUNT(*) AS total_ticks FROM {full_table_path}"))